# DHBW Machine Learning: R2-D2

In [73]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import KFold, cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures
from sklearn.tree import DecisionTreeRegressor

## Buisiness & Data Understanding

### Überblick über den Datensatz

In [74]:
data = pd.read_csv("SeoulBikeData.csv")
data.head()

,Date,Rented Bike Count,Hour,Temperature(°C),Humidity(%),Wind speed (m/s),Visibility (10m),Dew point temperature(°C),Solar Radiation (MJ/m2),Rainfall(mm),Snowfall (cm),Seasons,Holiday,Functioning Day
0,01/12/2017,254,0,-5.2,37,2.2,2000,-17.6,0.0,0.0,0.0,Winter,No Holiday,Yes
1,01/12/2017,204,1,-5.5,38,0.8,2000,-17.6,0.0,0.0,0.0,Winter,No Holiday,Yes
2,01/12/2017,173,2,-6.0,39,1.0,2000,-17.7,0.0,0.0,0.0,Winter,No Holiday,Yes
3,01/12/2017,107,3,-6.2,40,0.9,2000,-17.6,0.0,0.0,0.0,Winter,No Holiday,Yes
4,01/12/2017,78,4,-6.0,36,2.3,2000,-18.6,0.0,0.0,0.0,Winter,No Holiday,Yes


## Data Preparation

### Umgang mit fehlenden Werten

In diesem Datensatz sind keine fehlende Werte vorhanden.

In [75]:
data.isna().values.any()

np.False_

### Codierung von Variablen
Date wird entfernt, da relevante Informationen durch `Temperatur`,`Season` und `Holiday` gegeben sind.

`Season`,`Holiday` und `Functioning Day` werden mit Dummy Variablen initialisiert.

In [76]:
data = data.drop(columns="Date")

In [77]:
data_encoded = pd.get_dummies(
    data=data, columns=["Seasons", "Holiday", "Functioning Day"], drop_first=True
)

### Zielvariable
Die Zielvariable soll die ausgeliehenen Fahrräder an einem gegeben Tag sein.

In [78]:
target = "Rented Bike Count"
X = data_encoded.drop(columns=target)
y = data_encoded[target]

### Aufteilung in Trainings- und Testdaten


In [79]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

## Modeling
### Linear Regression

In [80]:
kfold = KFold(n_splits=10, shuffle=True, random_state=42)
lr_model = LinearRegression()
cv_scores = cross_val_score(
    lr_model, X_train, y_train, cv=kfold, scoring="neg_mean_squared_error"
)
cv_error = -cv_scores.mean()
np.sqrt(cv_error)

np.float64(431.62529986615147)

### Polynomiale Regression


In [81]:
polypipe = Pipeline([
    ("polynomial", PolynomialFeatures(degree=2, include_bias=False)),
    ("linear_regression", LinearRegression())
])


In [82]:
kfold = KFold(n_splits=10, shuffle=True, random_state=42)
cv_scores = cross_val_score(
    polypipe, X_train, y_train, cv=kfold, scoring="neg_mean_squared_error"
)
cv_error = -cv_scores.mean()
np.sqrt(cv_error)

np.float64(352.7958884382892)

### Regression Tree

In [83]:
tree = DecisionTreeRegressor(
    random_state = 42 
) 

In [84]:
kfold = KFold(n_splits=10, shuffle=True, random_state=42)
cv_scores = cross_val_score(
    tree, X_train, y_train, cv=kfold, scoring="neg_mean_squared_error"
)
cv_error = -cv_scores.mean()
np.sqrt(cv_error)

np.float64(323.2129488095643)

TypeError: BaseDecisionTree.decision_path() missing 1 required positional argument: 'X'